# ⚠️ IMPORTANT: Simulated Data Notice

**The cached responses in `results/mock_cache/` are SIMULATED, not real LLM outputs.**

They were generated using a random number generator to demonstrate the pipeline without requiring an API key.

**The results shown below do NOT represent real model performance.** They are synthetic examples for demonstration only.

---

# Evaluating LLM Prompt Strategies for Sentiment Classification

**Research Question:** How do different prompting strategies affect the accuracy, consistency, and cost of LLM-based sentiment classification?

This notebook demonstrates a framework for evaluating four prompting strategies:
- **Zero-shot**: Direct classification without examples
- **Few-shot**: Classification with 3 labelled examples
- **Chain-of-thought**: Explicit reasoning before classification
- **Structured JSON**: Requesting structured output format

All experiments use fixed random seeds and cached responses for full reproducibility.

## Setup

This notebook runs in **offline mode** using simulated responses, so no API key is required.

**To obtain real results:**
1. Copy `.env.example` to `.env`
2. Add your `OPENAI_API_KEY`
3. Set `OFFLINE_MODE = False` below
4. Run all cells to generate real cached responses

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

# Add src to path
sys.path.insert(0, str(Path.cwd()))

from src import create_client, compare_strategies, get_dataset

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

print("✓ Setup complete")

## Configuration

In [ ]:
# Experiment configuration
OFFLINE_MODE = True  # Set to False to make real API calls
MODEL = "gpt-3.5-turbo"
SEEDS = [42, 43, 44]  # Multiple seeds for consistency measurement

# Strategies to evaluate
STRATEGIES = [
    "zero_shot",
    "few_shot",
    "chain_of_thought",
    "structured_json"
]

if OFFLINE_MODE:
    print("⚠️  Running in OFFLINE MODE with SIMULATED data")
    print("   Results do NOT represent real model performance.")
else:
    print("✓ Running in ONLINE MODE with real API calls")
    
print(f"Model: {MODEL}")
print(f"Seeds: {SEEDS}")
print(f"Strategies: {', '.join(STRATEGIES)}")

## Dataset

We use a hand-crafted dataset of 100 short sentences (50 positive, 50 negative) for educational purposes.

In [ ]:
texts, labels = get_dataset()

print(f"Dataset size: {len(texts)} examples")
print(f"Class distribution: {sum(labels)} positive, {len(labels) - sum(labels)} negative")
print(f"\nExample texts:")
for i in range(3):
    label_str = "positive" if labels[i] == 1 else "negative"
    print(f"  [{label_str}] {texts[i]}")

## Evaluation

We evaluate each strategy across multiple random seeds to measure both accuracy and consistency.

In [ ]:
# Create LLM client
client = create_client(offline_mode=OFFLINE_MODE, model=MODEL)

# Run evaluation
results = compare_strategies(
    client=client,
    strategy_names=STRATEGIES,
    seeds=SEEDS,
    verbose=True
)

print("\n✓ Evaluation complete")

if OFFLINE_MODE:
    print("\n⚠️  Remember: These are SIMULATED results for demonstration only.")

## Results

⚠️ **If running in offline mode, these are simulated results for demonstration only.**

In [ ]:
# Extract metrics into a DataFrame
metrics_data = []
for strategy, data in results.items():
    m = data['metrics']
    metrics_data.append({
        'Strategy': strategy.replace('_', ' ').title(),
        'Accuracy': f"{m['accuracy']:.3f}",
        'Consistency': f"{m['consistency']:.3f}",
        'Parse Rate': f"{m['parse_rate']:.3f}",
    })

df_metrics = pd.DataFrame(metrics_data)
print("\n=== Performance Metrics ===")
if OFFLINE_MODE:
    print("⚠️  SIMULATED DATA - Not real model performance\n")
print(df_metrics.to_string(index=False))

### Visualisations

In [ ]:
# Accuracy comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
strategies_clean = [s.replace('_', ' ').title() for s in STRATEGIES]
accuracies = [results[s]['metrics']['accuracy'] for s in STRATEGIES]

axes[0].bar(strategies_clean, accuracies, color='steelblue', alpha=0.8)
axes[0].set_ylabel('Accuracy', fontsize=12)
title = 'Classification Accuracy by Strategy'
if OFFLINE_MODE:
    title += ' (Simulated Data)'
axes[0].set_title(title, fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].axhline(y=0.5, color='red', linestyle='--', alpha=0.3, label='Random baseline')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=15)

# Consistency
consistencies = [results[s]['metrics']['consistency'] for s in STRATEGIES]

axes[1].bar(strategies_clean, consistencies, color='coral', alpha=0.8)
axes[1].set_ylabel('Consistency', fontsize=12)
title = f'Consistency Across {len(SEEDS)} Seeds'
if OFFLINE_MODE:
    title += ' (Simulated Data)'
axes[1].set_title(title, fontsize=14, fontweight='bold')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## Discussion

### Framework Demonstration

This notebook demonstrates:
1. **Systematic evaluation** of multiple prompting strategies
2. **Consistency measurement** across random seeds
3. **Reproducible caching** for exact result replication
4. **Comprehensive metrics** including accuracy, consistency, and parse rates

### ⚠️ Data Validity Notice

If you ran this notebook in offline mode with the included mock cache:
- **The numbers shown are simulated, not real model measurements**
- They demonstrate the pipeline but do not represent actual LLM behaviour
- To obtain real research findings, run with `OFFLINE_MODE = False` and your API key

### Threats to Validity (For Real Evaluations)

When running with real API calls:

- **Small dataset**: 100 examples may not generalise to broader domains
- **Simple task**: Binary sentiment is relatively easy; complex tasks may show different patterns
- **Single model**: Results are specific to the model used
- **Hand-crafted data**: Real-world text may contain more ambiguity and edge cases
- **Limited seeds**: Only 3 seeds tested for consistency
- **Parsing assumptions**: Response parsing may miss valid but differently-formatted answers

### Next Steps

1. **Run with real API**: Set up your API key and run with real models
2. **Scale up**: Test on larger, real-world datasets (e.g., full SST-2, IMDB reviews)
3. **More strategies**: Self-consistency, role prompting, retrieval-augmented generation
4. **Model comparison**: GPT-4, Claude, Llama, open-source models
5. **Error analysis**: Characterise failure cases and error patterns
6. **Parameter sweep**: Temperature, top-p, max tokens effects

### References

- Brown et al. (2020). "Language Models are Few-Shot Learners." NeurIPS.
- Wei et al. (2022). "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models." NeurIPS.
- Kojima et al. (2022). "Large Language Models are Zero-Shot Reasoners." NeurIPS.
- Wang et al. (2022). "Self-Consistency Improves Chain of Thought Reasoning in Language Models." ICLR 2023.

## Reproduction Notes

This framework is designed for full reproducibility:

- **Fixed seeds**: All API calls use deterministic seeds (42, 43, 44)
- **Cached responses**: Results are cached by (model, prompt, seed) tuple
- **Offline mode**: The notebook can run without API access using cached responses
- **Version pinning**: `requirements.txt` pins all dependency versions

Once you run a real evaluation:
- Your results will be cached for exact reproduction
- Others can verify by inspecting your cache directory
- Re-running with same seeds will use cache (no new API calls)

⚠️ **The included mock cache demonstrates the reproducibility mechanism but does not contain real research data.**